### Actual implementation file

### RAG Pipelines -  Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

d:\Computer Science\Switch\Gen AI\RAG\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
d:\Computer Science\Switch\Gen AI\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Resume_BTech.pdf
  ✓ Loaded 1 pages

Processing: Resume_RA.pdf
  ✓ Loaded 1 pages

Total documents loaded: 2


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-13T16:10:25+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-13T16:10:25+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf_files\\Resume_BTech.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Resume_BTech.pdf', 'file_type': 'pdf'}, page_content='Mohammad Ausaf Shah\n+91-9682138656 | ms2396@srmist.edu.in | Linkdin | Github\nEducation\nSRM Institute of Science and T echnology Kattankulathur, Tamil Nadu\nB.Tech in Computer Science and engineering - 9.11 CGPA July. 2020 – Currently\nBurn Hall School Srinagar, J&K\nClass XII - 92.2% 2019 – 2020\nBurn Hall School Srinagar, J&K\nClass X - 92% 2017 – 2018\nExperience\nW eb Development Internship February 2022 – March 2022\nBooksApp Remote\n∗ Developed a landing page fo

In [4]:
### Text splitting get into chunks
# Its job is to take massive files and chop them into manageable pieces so the AI doesn't get overwhelmed or lose context.

def split_documents(documents,chunk_size=1000,chunk_overlap=200): 
    # chunk_size: The target length of each piece(chunk) (usually measured in characters or tokens). 1,000 is a standard "sweet spot" for many models.
    # chunk_overlap: This creates a "sliding window." It keeps 200 characters from the end of one chunk at the start of the next. This prevents the splitter from cutting a sentence in half and losing the meaning.
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter( # class object
        #This initializes the logic. The Recursive part is key: it tries to split by the first separator, and if the resulting piece is still too big, it moves to the next separator in the list.
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len, # Tells the splitter to count the size based on Python's standard string length.
        separators=["\n\n", "\n", " ", ""] # This is the priority list. It tries to split at double line breaks first (paragraphs), then single lines, then spaces (words), and finally individual characters if all else fails. This keeps related text together.
    )
    split_docs = text_splitter.split_documents(documents)
    # When you run text_splitter.split_documents(documents), you aren't calling your own function again. Instead, you are calling a built-in method that belongs to the RecursiveCharacterTextSplitter class.
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 2 documents into 8 chunks

Example chunk:
Content: Mohammad Ausaf Shah
+91-9682138656 | ms2396@srmist.edu.in | Linkdin | Github
Education
SRM Institute of Science and T echnology Kattankulathur, Tamil Nadu
B.Tech in Computer Science and engineering - ...
Metadata: {'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-13T16:10:25+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-13T16:10:25+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf_files\\Resume_BTech.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Resume_BTech.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-13T16:10:25+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-13T16:10:25+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf_files\\Resume_BTech.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Resume_BTech.pdf', 'file_type': 'pdf'}, page_content='Mohammad Ausaf Shah\n+91-9682138656 | ms2396@srmist.edu.in | Linkdin | Github\nEducation\nSRM Institute of Science and T echnology Kattankulathur, Tamil Nadu\nB.Tech in Computer Science and engineering - 9.11 CGPA July. 2020 – Currently\nBurn Hall School Srinagar, J&K\nClass XII - 92.2% 2019 – 2020\nBurn Hall School Srinagar, J&K\nClass X - 92% 2017 – 2018\nExperience\nW eb Development Internship February 2022 – March 2022\nBooksApp Remote\n∗ Developed a landing page fo

### Embedding and VectorStore DB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer # the embedding model will be available insie this
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity # cosine similarity will be used while retrieving from the vector database

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): # in this init method we are initializing the embedding manager and loading the model. The model name that we are giving is "all-MiniLM-L6-v2" which is a popular model for generating sentence embeddings, it is available fromHuggingFace. It converts text into vectors and we get around 384 dimensions
        """
        Initialize the embedding manager.

        Args:
            model_name: HuggingFace model name for SentenceTransformer 
        """
        self.model_name = model_name # Initialising the model name
        self.model = None
        self._load_model() 
    
    def _load_model(self): # the _ means protected function, it will be called only in this class
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model :{self.model_name}")
            self.model = SentenceTransformer(self.model_name) # loading the model using sentence transformer
            print(f"Model loaded successfully. Embedding dimension:{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:  # this method takes text, which is basically List of strings and finally returns a numpy array.
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed.

        Returns:
            Numpy array of embeddings with shape (len(texts), embedding_dimension).
        """
        if not self.model:
            raise ValueError("Model not loaded.")
        
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True) # generating the embeddings using the model
            print(f"Generated embeddings generated successfully with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise

    def get_embedding_dimension(self) -> int:
        """Get embedding dimension of the model."""
        if not self.model:
            raise ValueError("Model not loaded.")
        return self.model.get_sentence_embedding_dimension()
    

### initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model :all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4670.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension:384


### Vector Store

In [8]:
import os
import chromadb
class VectoreStore:
    """Manages document embeddings using ChromaDB vector store."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"): ## We are giving the collection name and persistant directory for the vector store. Persistant_directory means whatever vectoreStore is created, we will store it in the hard disk.
        """
        Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection.
            persist_directory: Directory to persist the ChromaDB collection.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store() ## Means this function will initialise the vector store
    
    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True) # We are specifying the durectory. If it alreadys exists, fine. Otherwise it will create it
            self.client = chromadb.PersistentClient(path=self.persist_directory) # We are creating a client which will have reference to the Chromadb VectorStore

            # Get or create collection
            self.collection = self.client.get_or_create_collection( # Creating the collection. Basically, where we will store the vectors inside the VectorStore
                name=self.collection_name,
                metadata={"description": "PDF Document embedding for RAG"}
            )
            print(f"Vector store initialized. Collection:{self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing VectorStore: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray): # The embedding is coming from generate_embeddings method of the embedding manager
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain documents.
            embeddings: Corresponding embeddings for the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
    
        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        document_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            document_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist()) 
        
        # Add to ChromaDB collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=document_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
    
vectorstore = VectoreStore()
vectorstore
        

Vector store initialized. Collection:pdf_documents
Existing documents in collection: 40


In [9]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-13T16:10:25+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-13T16:10:25+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf_files\\Resume_BTech.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Resume_BTech.pdf', 'file_type': 'pdf'}, page_content='Mohammad Ausaf Shah\n+91-9682138656 | ms2396@srmist.edu.in | Linkdin | Github\nEducation\nSRM Institute of Science and T echnology Kattankulathur, Tamil Nadu\nB.Tech in Computer Science and engineering - 9.11 CGPA July. 2020 – Currently\nBurn Hall School Srinagar, J&K\nClass XII - 92.2% 2019 – 2020\nBurn Hall School Srinagar, J&K\nClass X - 92% 2017 – 2018\nExperience\nW eb Development Internship February 2022 – March 2022\nBooksApp Remote\n∗ Developed a landing page fo

In [10]:
### Convert the text to Embeddings

texts = [doc.page_content for doc in chunks] # we are taking the page content from the chunks and creating a list of texts

texts

['Mohammad Ausaf Shah\n+91-9682138656 | ms2396@srmist.edu.in | Linkdin | Github\nEducation\nSRM Institute of Science and T echnology Kattankulathur, Tamil Nadu\nB.Tech in Computer Science and engineering - 9.11 CGPA July. 2020 – Currently\nBurn Hall School Srinagar, J&K\nClass XII - 92.2% 2019 – 2020\nBurn Hall School Srinagar, J&K\nClass X - 92% 2017 – 2018\nExperience\nW eb Development Internship February 2022 – March 2022\nBooksApp Remote\n∗ Developed a landing page for a Cargo company based in Dubai using HTML, CSS, JavaScript and Bootstrap.\n∗ Worked on the Navbar and Different Sections of the Website.\n∗ Attended daily reviews with the manager of BooksApp.\nProjects\nChat Mobile App | Flutter, Dart, Firebase, Firestore database August 2023 – August 2023\nGithub Link\n∗ Developed a Full stack Chatting Mobile App where a person can Login/ Sign up, upload a photo and can start\nchatting. It also has the functionality of push notifications and user authentication.',
 '∗ Developed a F

In [11]:
### Generating the embeddings of the above chunks
embeddings = embedding_manager.generate_embeddings(texts)


### Add the document chunks and their embeddings to the vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 8 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.79it/s]

Generated embeddings generated successfully with shape: (8, 384)
Adding 8 documents to the vector store...
Successfully added 8 documents to the vector store.
Total documents in collection after addition: 48


### Retriever Pipeline for VectorStore

In [12]:
### The Retriever is basically an interface that is connected to the vectorStore. 
# It takes a query, converts it into an embedding using the embedding manager, and then searches the vector store for the most similar document chunks. 
# The retrieved chunks can then be used to provide context to the language model when answering questions.

class RAGRetriever:
    """Handles quer-based retrieval from the vectorStore"""

    def __init__(self, vector_store: VectoreStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RAG retriever.

        Args:
            vector_store: Vector Store containing document embeddings
            embedding_manager: Manager for generating query embeddings.
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = -50.0) -> List[Dict[str, Any]]: ## Krish had kept the score_threshold: float = 0.0, but for me the similarity score came below it for 2 resumes, and then I made it low
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [13]:
rag_retriever

In [14]:
rag_retriever.retrieve("What is the nodejs project ?")

Retrieving documents for query: 'What is the nodejs project ?'
Top K: 5, Score threshold: -50.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 100.52it/s]

Generated embeddings generated successfully with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_7543b2f3_6',
  'content': 'HTML, CSS, and Bootstrap, resulting in a 14% increase in contracts generated through the website.\nSKILLS\nLanguagesJavaScript, Java, C++, SQL\nDatabasesMongoDB, PostgreSQL, Firebase\nF rameworks & LibrariesSpring Boot, Spring Reactive (WebFlux), React.js, Node.js, Express.js\nT ools & PlatformsGenAI, Prompt Engineering, MCP Servers, RAG, JMeter, GitLab, CI/CD, Jenkins,\nPostman, Jira, ChatGPT, Copilot, Gemini, Perplixity\nConceptsData Structures & Algorithms, Operating Systems, Object Oriented Programming,\nDatabase Management Systems, RESTful APIsPERSONAL PROJECT\nFinDash|React.js, Node.js, Express.js, MongoDB, Mongoose, Machine Learning[UI Code] [BE Code]\n•Built the Node.js backend of a full-stack finance dashboard featuring interactive charts, tables, and scatter plots\nto visualize revenue, expenses, profits, product prices, and transaction data.\n•Also has a predictions page powered by a linear regression ML model to forecast annual revenu

### Integration - VectorDB context pipeline with LLM Output

In [15]:
import os
from dotenv import load_dotenv 
load_dotenv() # to load the environment variables from the .env file, basically where I have the API key for the LLM

#print(os.getenv("GROQ_API_KEY"))

True

In [16]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [17]:
groq_api_key = "" # add your API key here

llm = ChatGroq(api_key=groq_api_key, model="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)


##  Simple RAG function : retrieve context and generate answer

def rag_simple(query, retriever, llm, top_k=3):
    ## retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context= "\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant information found in the documents."
    

    ## generate the answer using Groq LLM
    prompt=f"""Use the following context to answer the question concisely.
            Context:
            {context}
            Question: {query}
            Answer:"""
    
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content
    

In [18]:
answer = rag_simple("What is the nodejs project about?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is the nodejs project about?'
Top K: 3, Score threshold: -50.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 75.65it/s]

Generated embeddings generated successfully with shape: (1, 384)
Retrieved 3 documents (after filtering)


The Node.js project is called FinDash, a full-stack finance dashboard featuring interactive charts, tables, and scatter plots to visualize revenue, expenses, profits, product prices, and transaction data. It also includes a predictions page powered by a linear regression ML model to forecast annual revenue, supporting strategic planning.


### Enhanced RAG Pipeline features

In [19]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is the nodejs project about?", rag_retriever, llm, top_k=3, min_score=-50, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is the nodejs project about?'
Top K: 3, Score threshold: -50
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 100.81it/s]

Generated embeddings generated successfully with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The Node.js project is called FinDash, a full-stack finance dashboard featuring interactive charts, tables, and scatter plots to visualize revenue, expenses, profits, product prices, and transaction data. It also includes a predictions page powered by a linear regression ML model to forecast annual revenue, supporting strategic planning.
Sources: [{'source': 'Resume_RA.pdf', 'page': 0, 'score': -0.13353538513183594, 'preview': 'HTML, CSS, and Bootstrap, resulting in a 14% increase in contracts generated through the website.\nSKILLS\nLanguagesJavaScript, Java, C++, SQL\nDatabasesMongoDB, PostgreSQL, Firebase\nF rameworks & LibrariesSpring Boot, Spring Reactive (WebFlux), React.js, Node.js, Express.js\nT ools & PlatformsGenAI, Pr...'}, {'source': 'Resume_RA.pdf', 'page': 0, 'score': -0.13353538513183594, 'preview': 'HTML, CSS, and Bootstrap, resulting in a 14% increase in contracts generated through the website.\nSKILLS\nLanguagesJavaScript, Java, C++, SQL\nDatabasesMongoDB, Post

In [20]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is the nodejs project ?", top_k=3, min_score=-50, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is the nodejs project ?'
Top K: 3, Score threshold: -50
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 93.15it/s]

Generated embeddings generated successfully with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
HTML, CSS, and Bootstrap, resulting in a 14% increase in contracts generated through the website.
SKILLS
LanguagesJavaScript, Java, C++, SQL
DatabasesMongoDB, PostgreSQL, Firebase
F rameworks & LibrariesSpring Boot, Spring Reactive (WebFlux), React.js

, Node.js, Express.js
T ools & PlatformsGenAI, Prompt Engineering, MCP Servers, RAG, JMeter, GitLab, CI/CD, Jenkins,
Postman, Jira, ChatGPT, Copilot, Gemini, Perplixity
ConceptsData Structures & Algorithms, Operating Systems, Object Oriented Programming,
Database Management Systems, RESTful APIsPERSONAL PROJECT
FinDash|React.js, Node.js, Express.js, MongoDB, Mongoose, Machine Learning[UI Code] [BE Code]
•Built the Node.js backend of a full-stack finance dashboard featuring interactive charts, tables, and scatter plots
to visualize revenue, expenses, profits, product prices, and transaction data.
•Also has a predictions page powered by a linear regression ML model to forecast annual revenue, supporting
strategic planning.
ACHIEVEMENT

HTML, CSS, and Bootstrap, resulting in a 14% increase in contracts generated through the website.
SKILLS
LanguagesJavaScript, Java, C++, SQL
DatabasesMongoDB, PostgreSQL, Firebase
F rameworks & LibrariesSpring Boot, Spring Reactive (WebFlux), React.js, Nod